In [ ]:
# 1) Clean install
# !pip uninstall -y pyspark findspark -q
!apt-get update -qq
!apt-get install -y openjdk-17-jdk-headless -qq
!pip install -q pyspark

In [ ]:
# 2) Set Java path
import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"

# Optional: check Java
!java -version
!echo $JAVA_HOME

In [ ]:
# 3) Start Spark
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("ColabSpark")
    .master("local[*]")
    .config("spark.ui.port", "4050")
    .getOrCreate()
)

sc = spark.sparkContext
print("Spark version:", spark.version)
print("SparkContext OK:", sc.appName)

In [ ]:
spark


In [ ]:
!pip install PyDrive
from pydrive.auth import GoogleAuth
from pydrive.drive import GoogleDrive
from google.colab import auth
from oauth2client.client import GoogleCredentials

# Authenticate and create the PyDrive client
auth.authenticate_user()
gauth = GoogleAuth()
gauth.credentials = GoogleCredentials.get_application_default()
drive = GoogleDrive(gauth)

In [ ]:
!pip install -q gdown

!gdown --id 10SAUfsDwyTFcXjNewbYhqtJ6q44HoJ1O -O Bombing_Operations.json.gz
!gdown --id 1KKWvQIN96F0u8PXu4T50c5Sei7K5wTCB -O Aircraft_Glossary.json.gz

# Guerra de Vietnam

**Pres. Johnson**: _What piensas de esto de Vietnam? Me gustaría escucharte hablar un poco bit._

**Senador Russell**: _Well, francamente, Señor Presidente, es el maldito desastre peor que he visto, y no me gusta presumir y nunca he tenido razón muchas veces en mi vida, pero sabía que íbamos a meternos en este tipo de lío cuando entramos there._

27 de mayo de 1964

![banner](https://raw.githubusercontent.com/epfl-ada/2019/c17af0d3c73f11cb083717b7408fedd86245dc4d/Tutorials/04%20-%20Scaling%20Up/img/banner.jpg)

----

La Guerra de Vietnam, también conocida como la Segunda Guerra de Indochina, y en Vietnam como la Guerra de Resistencia contra Estados Unidos o simplemente la Guerra de los Estados Unidos, fue un conflicto que ocurrió en Vietnam, Laos y Camboya desde el 1 de noviembre de 1955 hasta la caída de Saigón el 30 de abril de 1975. Fue la segunda de las Guerras de Indochina y se libró oficialmente entre Vietnam del Norte y el gobierno de Vietnam del Sur.

**El conjunto de datos describe toda la operación de la fuerza aérea durante la Guerra de Vietnam.**

**Bombing_Operations** [Get the dataset here](https://drive.google.com/a/epfl.ch/file/d/1L6pCQkldvdBoaEhRFzL0VnrggEFvqON4/view?usp=sharing)

- AirCraft: _Aircraft model (example: EC-47)_
- ContryFlyingMission: _Country_
- MissionDate: _Date of the mission_
- OperationSupported: _Supported War operation_ (example: [Operation Rolling Thunder](https://en.wikipedia.org/wiki/Operation_Rolling_Thunder))
- PeriodOfDay: _Day or night_
- TakeoffLocation: _Take off airport_
- TimeOnTarget
- WeaponType
- WeaponsLoadedWeight

**Aircraft_Glossary** [Get the dataset here](https://drive.google.com/a/epfl.ch/file/d/14dyBmcTBA32uXPxDbqr0bFDIzGxMTWwl/view?usp=sharing)

- AirCraft: _Aircraft model (example: EC-47)_
- AirCraftName
- AirCraftType

**Información del dataset:**

THOR es una base de datos cuidadosamente cultivada de bombardeos aéreos históricos desde la Primera Guerra Mundial hasta Vietnam. THOR ya ha demostrado ser útil para encontrar municiones sin explotar en el sudeste asiático y mejorar las tácticas de combate de la Fuerza Aérea:

https://www.kaggle.com/usaf/vietnam-war-bombing-operations

Cargue el datasets:

In [ ]:
Bombing_Operations = spark.read.json("Bombing_Operations.json.gz")
Aircraft_Glossary = spark.read.json("Aircraft_Glossary.json.gz")

Compruebe el esquema:

In [ ]:
Bombing_Operations.printSchema()

In [ ]:
Aircraft_Glossary.printSchema()

Obtenga una muestra con `take()`:

In [ ]:
Bombing_Operations.take(2)

Obtener una muestra formateada con `show()`:

In [ ]:
Aircraft_Glossary.show()

In [ ]:
print("In total there are {0} operations".format(Bombing_Operations.count()))

## Pregunta 1: ¿Qué países están involucrados y en cuántas misiones?
Palabras clave: `Dataframe API`, `SQL`, `group by`, `sort`

Vamos a agrupar las misiones por 'ContryFlyingMission' y contar cuántos registros existen:

In [ ]:
Bombing_Operations.show()

In [ ]:
from pyspark.sql.functions import desc
tmp1 = Bombing_Operations.groupBy("ContryFlyingMission").count().sort(desc("count"))
tmp1.show()

In [ ]:
from pyspark.sql.functions import count, desc
missions_counts = Bombing_Operations.groupBy("ContryFlyingMission")\
                                    .agg(count("*").alias("MissionsCount"))\
                                    .sort(desc("MissionsCount"))
missions_counts.show()

En este caso utilizamos la API DataFrame, pero podríamos rewitear el `groupBy` usando SQL puro:

In [ ]:
Bombing_Operations.registerTempTable("Bombing_Operations")

query = """
SELECT ContryFlyingMission, count(*) as MissionsCount
FROM Bombing_Operations
GROUP BY ContryFlyingMission
ORDER BY MissionsCount DESC
"""

missions_counts = spark.sql(query)
missions_counts.show()

El Dataframe es lo suficientemente pequeño como para ser trasladado a Pandas:

In [ ]:
missions_count_pd = missions_counts.toPandas()
missions_count_pd.head()

Vamos a trazar un gráfico de barras con el número de misiones por país:

In [ ]:
pl = missions_count_pd.plot(kind="bar",
                            x="ContryFlyingMission", y="MissionsCount",
                            figsize=(10, 7), log=True, alpha=0.5, color="red")
pl.set_xlabel("Country")
pl.set_ylabel("Number of Missions (Log scale)")
pl.set_title("Number of missions by Country")


----

## Preguntas 2: Muestre el número de misiones a tiempo para cada uno de los países involucrados.

Palabras clave: `group by`, `parse date`, `plot`

Seleccionemos las columnas relevantes:

In [ ]:
missions_countries = Bombing_Operations.selectExpr(["to_date(MissionDate) as MissionDate", "ContryFlyingMission"])
missions_countries.show()

El MissionDate archivado se convierte en un objeto `date` de Python.

Ahora podemos agrupar por `MissionDate` y `ContryFlyingMission` para obtener el recuento:

In [ ]:
from pyspark.sql.functions import count, asc
missions_by_date = missions_countries\
                    .groupBy(["MissionDate", "ContryFlyingMission"])\
                    .agg(count("*").alias("MissionsCount"))\
                    .sort(asc("MissionDate"))
missions_by_date.show()

Ahora podemos trazar el contenido con una serie diferente para cada país:

In [ ]:
import matplotlib.pyplot as plt

missions_by_date_pd = missions_by_date.toPandas()
missions_by_date_pd.head()
fig = plt.figure(figsize=(10, 6))

# iterate the different groups to create a different series
for country, missions in missions_by_date_pd.groupby("ContryFlyingMission"):
    plt.plot(missions["MissionDate"], missions["MissionsCount"], label=country)

plt.legend(loc='best')

Podemos observar cómo Vietnam del Sur aumentó sus misiones a partir de 1970. La caída en 1973 está motivada por los [Acuerdos de Paz de París] (https://en.wikipedia.org/wiki/Paris_Peace_Accords) que tuvieron lugar el 27 de enero de 1973, para establecer la paz en Vietnam y poner fin a la guerra.

----

## Pregunta 3: ¿Quién bombardeó este lugar?

Palabras clave: 'RDD map reduce' 'cache' 'save results'

<img style="float: right;" src="https://raw.githubusercontent.com/epfl-ada/2019/c17af0d3c73f11cb083717b7408fedd86245dc4d/Tutorials/04%20-%20Scaling%20Up/img/Hanoi_POL1966.jpg">

Esta imagen es la instalación POL de Hanoi (Vietnam del Norte) ardiendo después de que fue atacada por la Fuerza Aérea de los Estados Unidos el 29 de junio de 1966 en el contexto de la operación Rolling Thunder.

Estamos interesados en descubrir cuál fue el lugar de despegue más común durante ese día.

In [ ]:
jun_29_operations = Bombing_Operations.where("MissionDate = '1966-06-29' AND TargetCountry='NORTH VIETNAM'")
jun_29_operations.show()

¿Qué coutries programaron misiones ese día?

In [ ]:
jun_29_operations.groupBy("ContryFlyingMission").agg(count("*").alias("MissionsCount")).toPandas()

La mayor parte de la operación de ese día fue realizada por aviones estadounidenses.

In [ ]:
jun_29_operations.take(1)

Puede especificar para almacenar en caché el contenido en la memoria:

In [ ]:
jun_29_operations.cache()

Ahora puede contar el número de filas y mover el contenido a la memoria caché:

In [ ]:
%time jun_29_operations.count()

La segunda vez que el contenido se almacena en caché y la operación es mucho más rápida:

In [ ]:
%time jun_29_operations.count()

También puede guardar los resultados en un archivo...

In [ ]:
jun_29_operations.write.mode('overwrite').json("jun_29_operations.json")

... y lea del archivo:

In [ ]:
jun_29_operations = spark.read.json("jun_29_operations.json")

Podemos usar la api simple de DataFrame...

In [ ]:
TakeoffLocationCounts = jun_29_operations\
                            .groupBy("TakeoffLocation").agg(count("*").alias("MissionsCount"))\
                            .sort(desc("MissionsCount"))
TakeoffLocationCounts.show()

.... o el formato explícito Map/Reduce con RDD.

Primero emitimos un par en el formato (Ubicación, 1):

In [ ]:
all_locations = jun_29_operations.rdd.map(lambda row: (row.TakeoffLocation, 1))
all_locations.take(10)

Luego, sumamos los contadores en el paso de reducción y ordenamos por conteo:

In [ ]:
locations_counts_rdd = all_locations.reduceByKey(lambda a, b: a+b).sortBy(lambda r: -r[1])
locations_counts_rdd.take(3)

Ahora podemos convertir el RDD en dataframe asignando los pares a objetos de tipo `Row`

In [ ]:
from pyspark.sql import Row
locations_counts_with_schema = locations_counts_rdd.map(lambda r: Row(TakeoffLocation=r[0], MissionsCount=r[1]))
locations_counts = spark.createDataFrame(locations_counts_with_schema)
locations_counts.show()

<img style="float: right;" src="https://raw.githubusercontent.com/epfl-ada/2019/c17af0d3c73f11cb083717b7408fedd86245dc4d/Tutorials/04%20-%20Scaling%20Up/img/USS_Constellation.jpg">

Ese día el lugar de despegue más común fue el buque USS Constellation (CV-64). No podemos identificar unívocamente un lugar de despegue, pero podemos reducir los posibles candidatos. Pasos siguientes: explore la función TimeOnTarget.

_USS Constellation (CV-64), un superportaaviones de la clase Kitty Hawk, fue el tercer buque de la Armada de los Estados Unidos en ser nombrado en honor a la "nueva constelación de estrellas" en la bandera de los Estados Unidos. Uno de los barcos más rápidos de la Armada, como lo demuestra su victoria durante una carrera de grupos de batalla celebrada en 1985, fue apodado "Connie" por su tripulación y oficialmente como "America's Flagship"._

----

 Preguntas 4: ¿Cuál es el tipo de avión más utilizado durante la guerra de Vietnam (número de misiones)?

Keywords: `join` `group by`

Revisemos el contenido de `Aircraft_Glossary`:

In [ ]:
Aircraft_Glossary.show(5)

Estamos interesados en el archivo `AirCraftType`.

In [ ]:
Bombing_Operations.select("AirCraft").show(5)

Podemos unirnos en la columna `AirCraft` de ambos dataframes.

 Con Dataframe API:

In [ ]:
missions_joined = Bombing_Operations.join(Aircraft_Glossary,
                                          Bombing_Operations.AirCraft == Aircraft_Glossary.AirCraft)
missions_joined

Podemos seleccionar solo el campo que nos interesa:

In [ ]:
missions_aircrafts = missions_joined.select("AirCraftType")
missions_aircrafts.show(5)

Y por último podemos agrupar por 'AirCraftType' y contar:

In [ ]:
missions_aircrafts.groupBy("AirCraftType").agg(count("*").alias("MissionsCount"))\
                  .sort(desc("MissionsCount"))\
                  .show()

 podemos reescribir esto en SQL puro:

In [ ]:
Bombing_Operations.registerTempTable("Bombing_Operations")
Aircraft_Glossary.registerTempTable("Aircraft_Glossary")

query = """
SELECT AirCraftType, count(*) MissionsCount
FROM Bombing_Operations bo
JOIN Aircraft_Glossary ag
ON bo.AirCraft = ag.AirCraft
GROUP BY AirCraftType
ORDER BY MissionsCount DESC
"""

spark.sql(query).show()

Los aviones de tipo `Fighter Jet Bomber` participaron en la mayoría de las misiones en la guerra de Vietnam.

Nota: Este conjunto de datos requeriría una mayor limpieza y normalización. Ver `Fighter Jet Bomber`, `Jet Fighter Bomber`, `Fighter bomber jet`

# Task

In [ ]:
#Using the Bombing_Operations table, apply the MapReduce programming model in Spark RDDs to count the number of
#operations per country.

## Step 1: Convert the DataFrame to an RDD
bombing_Operations_rdd = Bombing_Operations.rdd

## Step 2: Apply MapReduce
Bomb_map = bombing_Operations_rdd.map(lambda r:(r.ContryFlyingMission, 1))
Bomb_map_reduced = Bomb_map.reduceByKey(lambda a, b: a+b)
df = Bomb_map_reduced.sortBy(lambda r: -r[1]).toDF(["ContryFlyingMission", "MissionsCount"])
## Step 3: Display the Top 10 Countries
df.show(10)